[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdn-cs502k-symbolic-ai/practicals/blob/main/day01/tutorial1-problem-formulation.ipynb)


# CS502K: Symbolic Artificial Intelligence

## Tutorial 01b: Problem Formulation

### SOLUTIONS

#### Prof. Felipe Meneguzzi

**This is the solutions notebook.** It carries a worked implementation of every exercise and an
answer to every discussion question. Release it after the practical session it belongs to.

This notebook is the second half of Tutorial 1, and follows Lecture 2.
Work through the paper tutorial first: it covers agents, rationality and PEAS,
and this notebook picks up where its last question leaves off.


In [ ]:
try:
    import google.colab
    print("We are in Google colab, we need to clone the repo")
    !git clone https://github.com/abdn-cs502k-symbolic-ai/practicals.git
    %cd practicals/day01
except:
    print("Not in colab")

# This notebook needs nothing else: everything it uses is defined below, so it
# also runs on its own if you downloaded it from MyAberdeen.

## Formulating a problem

A *search problem* is a description of a task precise enough for an algorithm to solve it
without knowing anything about the task. Lecture 2 defines one as five components:

| Component | What it says | In the code |
| --- | --- | --- |
| Initial state | where the agent starts | `Problem.initial` |
| Actions | what the agent can do in a state | `Problem.actions(state)` |
| Transition model | what each action does | `Problem.result(state, action)` |
| Goal test | whether a state is a goal | `Problem.goal_test(state)` |
| Path cost | what a sequence of actions costs | `Problem.path_cost(c, s1, a, s2)` |

The lecture writes the second and third together as the *successor function* $S(x)$, which returns
action-state pairs. The code splits that into `actions` and `result`, because an algorithm usually
wants to ask what it can do before working out what would happen.

A *solution* is a sequence of actions leading from the initial state to a goal state.

**What you write, and what we give you.** The cell below defines the `Problem` class you subclass,
and that is the whole of what we give you. There is no search algorithm anywhere in this notebook,
because you implement those in Tutorial 2 and handing you one here would hand you that answer too.

You therefore check a formulation without solving it, which is the honest test in any case. Apply
the transition model to a state and see whether you get the state you expected. Ask whether the
goal test fires where it should. Enumerate the states your formulation can reach and compare that
set against the one the lecture draws. A formulation that passes those checks is right whether or
not any algorithm has run on it.


### On coding assistants

A language model writes `VacuumWorld` in seconds, and writes it correctly. We know, and we wrote
the exercise anyway. What it asks of you is a judgement about representation: which details of a
task belong in a state, and which you can leave out. That judgement depends on what you mean to do
with the model afterwards, so an assistant that has not been told your intent cannot make it for
you.

The choice matters for how this course assesses you. Assessment 1 is invigilated and Assessment 2
is proctored, so you sit both with no model available. Assessment 3 allows generative AI, and ends
in a demonstration where you answer questions about what you submitted. None of the three gives
you credit for what you can do only with an assistant.

So use a model to explain a concept you have not understood, or to criticise a formulation you
have already written. If you paste the stub in and paste the answer back, your notebook will run
and you will still not know why, with eleven weeks left before the assessments test whether you do.


In [ ]:
# The Problem class you subclass to formulate a search problem. Read it: the
# five components of Section 3.1 are the five methods below.
#
# From aima-python, the reference implementation accompanying Russell and
# Norvig's *Artificial Intelligence: A Modern Approach*, used under the MIT
# license and trimmed to this one class. It carries no search algorithm: you
# implement those in Tutorial 2.


def is_in(elt, seq):
    """Similar to (elt in seq), but compares with 'is', not '=='."""
    return any(x is elt for x in seq)


class Problem:
    """The abstract class for a formal problem. You should subclass
    this and implement the methods actions and result, and possibly
    __init__, goal_test, and path_cost."""

    def __init__(self, initial, goal=None):
        """The constructor specifies the initial state, and possibly a goal
        state, if there is a unique goal. Your subclass's constructor can add
        other arguments."""
        self.initial = initial
        self.goal = goal

    def actions(self, state):
        """Return the actions that can be executed in the given state."""
        raise NotImplementedError

    def result(self, state, action):
        """Return the state that results from executing the given action in
        the given state. The action must be one of self.actions(state)."""
        raise NotImplementedError

    def goal_test(self, state):
        """Return True if the state is a goal. The default method compares the
        state to self.goal, or checks for state in self.goal if it is a list,
        as specified in the constructor. Override this method if checking
        against a single self.goal is not enough."""
        if isinstance(self.goal, list):
            return is_in(state, self.goal)
        else:
            return state == self.goal

    def path_cost(self, c, state1, action, state2):
        """Return the cost of a solution path that arrives at state2 from
        state1 via action, assuming cost c to get up to state1. The default
        method costs 1 for every step in the path."""
        return c + 1

## Part 1: read a formulation

Start with the 8-puzzle, which Lecture 2 uses as its worked example. A state is a tuple of nine
numbers giving the tile at each position, with `0` for the blank, so the goal

```
1 2 3
4 5 6
7 8 _
```

is the tuple `(1, 2, 3, 4, 5, 6, 7, 8, 0)`. The actions move the *blank*, not the tiles, which
is the abstraction the lecture makes when it says `move blank left, right, up, down'. Read the
code below and find each of the five components in it.


In [ ]:
class EightPuzzle(Problem):
    """Sliding tiles numbered 1 to 8 on a 3x3 board with one blank square."""

    def __init__(self, initial, goal=(1, 2, 3, 4, 5, 6, 7, 8, 0)):
        super().__init__(initial, goal)

    def actions(self, state):
        """Return the moves of the blank square that stay on the board."""
        possible = ['UP', 'DOWN', 'LEFT', 'RIGHT']
        blank = state.index(0)
        if blank % 3 == 0:
            possible.remove('LEFT')
        if blank < 3:
            possible.remove('UP')
        if blank % 3 == 2:
            possible.remove('RIGHT')
        if blank > 5:
            possible.remove('DOWN')
        return possible

    def result(self, state, action):
        """Swap the blank with its neighbour in the direction of the action."""
        blank = state.index(0)
        delta = {'UP': -3, 'DOWN': 3, 'LEFT': -1, 'RIGHT': 1}[action]
        new_state = list(state)
        neighbour = blank + delta
        new_state[blank], new_state[neighbour] = new_state[neighbour], new_state[blank]
        return tuple(new_state)


def show(state):
    """Print a 9-tuple as a 3x3 board."""
    for row in range(0, 9, 3):
        print(' '.join(str(t) if t else '_' for t in state[row:row + 3]))


In [ ]:
puzzle = EightPuzzle((1, 2, 3, 4, 5, 6, 0, 7, 8))
show(puzzle.initial)
print('\nactions here:', puzzle.actions(puzzle.initial))
print('goal already?', puzzle.goal_test(puzzle.initial))

# Apply a sequence of actions by hand and watch the state change.
state = puzzle.initial
for action in ['RIGHT', 'RIGHT']:
    state = puzzle.result(state, action)
    print(f'\nafter {action}:')
    show(state)

print('\ngoal reached?', puzzle.goal_test(state))

### Exercise 1

Answer these from the code above. No programming.

1. Which line gives the initial state? Which gives the goal test?
2. `EightPuzzle` defines `actions` and `result` but neither `goal_test` nor `path_cost`.
   Look at `Problem` in the cell above and say what it inherits in each case,
   and why those defaults suit this problem.
3. The actions move the blank rather than the tiles, and both descriptions generate the same
   successors from any state. So what does naming the actions after the blank actually buy you?
   Count the action names each description needs, and say what `actions` has to check in each.
4. `result` assumes its action is legal in the state it receives. Find the reason that assumption
   holds, and say which method would have to change if it did not.


### Answers to Exercise 1

**1. Initial state and goal test.** `EightPuzzle.__init__` hands `initial` to
`super().__init__(initial, goal)`, so `Problem.__init__` stores it as `self.initial`. Nothing in
`EightPuzzle` performs the goal test: it inherits `Problem.goal_test`, which compares the state
against `self.goal`, and `self.goal` took the default `(1, 2, 3, 4, 5, 6, 7, 8, 0)` from
`EightPuzzle.__init__`.

**2. What the subclass inherits.** It inherits `goal_test` and `path_cost`.

`goal_test` compares for equality against a single stored state. That suits the 8-puzzle because
the puzzle has exactly one goal configuration, so equality answers the question completely. The
vacuum world in Part 2 does not have that property, which is why Part 2 asks you to override it.

`path_cost` returns `c + 1`, charging one unit per action. That suits the puzzle because every
slide is the same amount of work, so the cost of a path equals the number of moves in it, and a
cheapest solution is a shortest solution.

**3. What naming actions after the blank buys.** It does not reduce the branching factor. Every
move of the blank *is* a move of exactly one tile, so the two descriptions generate the same
successors and each state has the same number of them either way.

What changes is the size of the action vocabulary and the work `actions` has to do. Naming the
blank needs four action names in total, `UP`, `DOWN`, `LEFT` and `RIGHT`, and `actions` decides
applicability by testing one index against the edges of the board. Naming the tiles needs a name
per tile and direction, up to 32 of them on a 3x3 board, and `actions` has to find the blank and
check adjacency for every tile before it can rule most of them out. The blank formulation also
carries over unchanged to a 4x4 or 5x5 puzzle, where the tile formulation grows.

**4. Why `result` may assume a legal action.** The `Problem` docstring states the contract: the
action passed to `result` must be one of `self.actions(state)`. `actions` has already removed the
moves that would take the blank off the board, so by the time `result` runs, `blank + delta` is
guaranteed to land on the board. Search algorithms respect that contract, because they only ever
reach `result` through `actions`.

If callers could pass any action at all, `result` is the method that would have to change: it
would need to validate its argument and raise or return the unchanged state. Note what goes wrong
without a guard. `blank + delta` would still index a list, because Python accepts negative
indices, so `LEFT` from the left-hand column would silently wrap the blank to the end of the
preceding row and produce a corrupt board rather than an error.

## Part 2: formulate the vacuum world

Now write a formulation of your own. Lecture 2 uses the two-room vacuum world throughout:
a robot occupies room `A` or room `B`, each room is either dirty or clean, and the robot
moves with `Left` and `Right` and cleans with `Suck`.

The lecture numbers the eight states 1 to 8. Odd-numbered states put the robot in room `A`,
even-numbered states put it in room `B`:

| State | Robot | Room A | Room B |
| ---: | :---: | :---: | :---: |
| 1 | A | dirty | dirty |
| 2 | B | dirty | dirty |
| 3 | A | dirty | clean |
| 4 | B | dirty | clean |
| 5 | A | clean | dirty |
| 6 | B | clean | dirty |
| 7 | A | clean | clean |
| 8 | B | clean | clean |

Represent a state as the tuple `(location, dirty_a, dirty_b)`, where `location` is `'A'` or `'B'`
and the other two are booleans. State 5 is therefore `('A', False, True)`, and the lecture gives
its solution as `[Right, Suck]`.

Fill in the three methods below.


In [ ]:
class VacuumWorld(Problem):
    """The two-room vacuum world of Lecture 2.

    A state is (location, dirty_a, dirty_b), e.g. ('A', False, True) for state 5.
    """

    def __init__(self, initial=('A', False, True)):
        # No goal state is passed: the goal is a condition, not a single state,
        # so we override goal_test below instead.
        super().__init__(initial)

    def actions(self, state):
        """Return the actions available in state.

        All three actions apply in every state, which is the choice the
        lecture's state-space graph makes: it draws an L self-loop on every
        state with the robot in A, an R self-loop on every state with the robot
        in B, and an S self-loop on every state whose current room is clean.
        Exercise 3 asks what the alternative choice would do.
        """
        return ['Left', 'Right', 'Suck']

    def result(self, state, action):
        """Return the state that results from taking action in state."""
        location, dirty_a, dirty_b = state
        if action == 'Left':
            return ('A', dirty_a, dirty_b)
        if action == 'Right':
            return ('B', dirty_a, dirty_b)
        if action == 'Suck':
            if location == 'A':
                return (location, False, dirty_b)
            return (location, dirty_a, False)
        raise ValueError(f'unknown action: {action}')

    def goal_test(self, state):
        """Return True when no room is dirty."""
        return not state[1] and not state[2]

### Exercise 2

Run the cell below. It applies your transition model to states whose successors the lecture
fixes, and it walks the plan the lecture gives for state 5 by hand. Nothing here solves the
problem: every check follows from your three methods alone. All six must pass.

In [ ]:
def check(condition, message):
    print(('PASS  ' if condition else 'FAIL  ') + message)
    return condition


world = VacuumWorld()

check(world.goal_test(('A', False, False)), 'a clean world is a goal')
check(not world.goal_test(('B', True, False)), 'a world with a dirty room is not a goal')

# Suck cleans the room the robot occupies, and moves nothing else.
check(world.result(('A', True, True), 'Suck') == ('A', False, True),
      'Suck in room A cleans A, leaves B dirty, leaves the robot in A')

# Sucking an already clean room is allowed and changes nothing.
check(world.result(('B', True, False), 'Suck') == ('B', True, False),
      'Suck in a clean room changes nothing')

# Moving changes the location and disturbs no dirt.
check(world.result(('A', True, False), 'Right') == ('B', True, False),
      'Right moves the robot to B and leaves the dirt alone')

# The lecture says state 5 solves as [Right, Suck]. Walk it by hand.
state = ('A', False, True)
for action in ['Right', 'Suck']:
    state = world.result(state, action)
check(world.goal_test(state),
      f"the lecture's plan [Right, Suck] reaches a goal from state 5, ended at {state}")

### Exercise 3: check your state space against the lecture

The cell below enumerates every state your formulation can reach from state 1, and every
transition between them. It does not search for a solution: it walks the whole space and records
what it finds. Compare the result with the state-space graph on the *Example: Vacuum world state
space graph* slide.

1. Do you reach all eight states? If not, which are missing, and is that a bug in your
   formulation or a property of the problem?
2. Count the transitions. If your count differs from the lecture's graph, work out which
   decision in `actions` accounts for the difference.
3. The lecture lists a fourth action, `NoOp`, with a path cost of 0. Say what adding it does to
   the state space, and what it would do to a search algorithm looking for a shortest plan.
4. Every state in this space can reach a goal. Argue that from the transition model alone,
   without running any search.

In [ ]:
def explore(problem):
    """Return the reachable states and transitions of a problem, breadth first."""
    seen, transitions, frontier = {problem.initial}, [], [problem.initial]
    while frontier:
        state = frontier.pop()
        for action in problem.actions(state):
            successor = problem.result(state, action)
            transitions.append((state, action, successor))
            if successor not in seen:
                seen.add(successor)
                frontier.append(successor)
    return seen, transitions


states, transitions = explore(VacuumWorld(('A', True, True)))
print(f'{len(states)} states reachable, {len(transitions)} transitions\n')
for state, action, successor in sorted(transitions, key=str):
    print(f'{str(state):24} --{action:6}-> {successor}')


### Answers to Exercise 3

The cell reports **8 states reachable, 24 transitions**, which matches the lecture's graph
exactly: eight boxes, each with three arrows leaving it.

**1. All eight states.** Yes, all eight are reachable from state 1. Moving reaches the other room
without touching dirt, and sucking clears the occupied room, so from both-rooms-dirty you can
reach every combination of dirt with the robot in either room.

Worth noticing in passing that the reverse does not hold. Start from a clean room and no action
makes it dirty again, so from state 7 or 8 only two states are reachable. The state space is
connected in the lecture's drawing only because the drawing shows the arcs, not the reachability
from one particular start.

**2. Counting the transitions.** Eight states with three actions each gives 24. The count follows
from the decision in `actions`, which offers all three actions everywhere. Had we offered only
the actions that change something, `Left` would disappear in room A, `Right` would disappear in
room B, and `Suck` would disappear wherever the current room is already clean, leaving 13
transitions instead of 24. Both formulations describe the same problem and admit the same
solutions; they differ only in whether the graph carries self-loops.

**3. Adding `NoOp` at cost 0.** It adds one self-loop to every state, so 32 transitions rather
than 24, and it changes nothing about which states are reachable or which plans solve the problem.

For a search algorithm it is a genuine hazard. A zero-cost action means a cycle whose total cost
is zero, so a plan can be padded with any number of `NoOp`s without its cost rising. Uniform-cost
search over a tree would sit on that cycle and never terminate, because the cheapest node on the
frontier keeps having the same cost forever. Graph search survives it, since the explored set
refuses a state it has already expanded, but the optimal solution is no longer unique in length
and the "shortest plan" is only shortest if you count actions rather than cost. This is why AIMA
requires that step costs be bounded below by some positive constant before uniform-cost search is
guaranteed complete.

**4. Every state reaches a goal, without searching.** Take any state. `Suck` leaves the occupied
room clean. A move changes only the location and never makes a room dirty. So the fixed sequence
`Suck`, then move to the other room, then `Suck` leaves both rooms clean from *any* starting
state, and it reaches a goal in at most three actions. The argument uses only the transition
model: sucking never dirties, and moving never dirties.

## Part 3: choosing the state space

Lecture 2 argues that a state space is an *abstraction*: an abstract state stands for a set of
real states, and an abstract action for a set of real action sequences. The vacuum world you just
wrote abstracts away everything except the room the robot occupies and whether each room is dirty.

### Exercise 4

1. Generalise your formulation to $n$ rooms in a row. Write `NVacuumWorld` below, taking the
   number of rooms as an argument, and check that two rooms still behave as before.
2. Give a formula for the number of states as a function of $n$, and confirm it with `explore`
   for $n = 2, 3, 4$.
3. Suppose each room also holds an amount of dirt between 0 and 9 rather than a yes-or-no flag.
   How many states now? Does any *solution* change? Say what this tells you about which details
   belong in a state space.
4. The lecture's *Selecting a state space* slide requires that every real state matching an
   abstract state can reach some real state matching the abstract successor. Give one detail of a
   real vacuum robot whose removal would break that requirement.


In [ ]:
class NVacuumWorld(Problem):
    """The vacuum world with n rooms in a row, numbered 0 to n-1."""

    def __init__(self, rooms=2, initial=None):
        self.rooms = rooms
        if initial is None:
            # The n-room analogue of state 1: robot at the left end, all dirty.
            initial = (0,) + (True,) * rooms
        super().__init__(initial)

    def actions(self, state):
        # All three everywhere, matching the two-room formulation above.
        return ['Left', 'Right', 'Suck']

    def result(self, state, action):
        location, dirt = state[0], list(state[1:])
        if action == 'Left':
            location = max(0, location - 1)          # a no-op at the left wall
        elif action == 'Right':
            location = min(self.rooms - 1, location + 1)
        elif action == 'Suck':
            dirt[location] = False
        else:
            raise ValueError(f'unknown action: {action}')
        return (location,) + tuple(dirt)

    def goal_test(self, state):
        return not any(state[1:])


# Two rooms must still behave as before: 8 states and 24 transitions.
states, transitions = explore(NVacuumWorld(2))
check(len(states) == 8 and len(transitions) == 24,
      f'two rooms still give 8 states and 24 transitions, got {len(states)} and {len(transitions)}')

print()
for n in (2, 3, 4):
    states, transitions = explore(NVacuumWorld(n))
    print(f'n = {n}:  {len(states):3} states, formula n * 2**n predicts {n * 2 ** n:3},'
          f'  {len(transitions):3} transitions')

### Answers to Exercise 4

**2. How many states.** A state fixes where the robot is, which is $n$ choices, and whether each
of the $n$ rooms is dirty, which is $2^{n}$ combinations. The two are independent, so there are
$n \cdot 2^{n}$ states. The cell above confirms 8, 24 and 64 for $n = 2, 3, 4$. For $n = 2$ this
reproduces the eight states of the lecture, which is the check worth doing before trusting the
formula.

**3. Dirt from 0 to 9.** Each room now has ten conditions rather than two, so the count becomes
$n \cdot 10^{n}$: for two rooms, 200 states in place of 8.

No solution changes. If `Suck` sets the room to 0 the shortest plan is exactly the plan you had
before, visiting each dirty room once and sucking. If instead `Suck` removes one unit of dirt, the
plans get longer but keep their shape, since the agent simply repeats `Suck` in place.

The lesson is about what earns a place in a state space. The extra detail multiplied the state
count by 25 and gave the agent nothing it can act on differently, because no decision it makes
depends on *how* dirty a room is, only on whether any dirt remains. A state space should record
what changes the choice of action, and discard the rest. That is what makes the abstraction in
Lecture 2 a design decision rather than a simplification for convenience.

**4. A detail whose removal breaks the abstraction.** The requirement is that any real state
matching the abstract state can reach some real state matching the abstract successor. A battery
breaks it. An abstract `Right` claims the robot can always cross to the next room, and a real
robot with a flat battery matches the same abstract state while being unable to move at all, so
the abstract action has no real counterpart from that real state. A door that can be shut between
the rooms breaks it the same way, as does a threshold the robot cannot climb. Anything that makes
a real move fail from some real states, while leaving the abstract description unchanged, breaks
the guarantee, and a plan found in the abstract space then fails to execute.

## Where this goes next

You have formulated two problems and run no search at all. Tutorial 2 supplies the other half:
you implement breadth-first search, depth-first search and A*, and you run them on the
formulations you wrote here. The vacuum world and the 8-puzzle come back as test cases, so a
formulation you get right this week is one you do not have to debug next week.